In [ ]:
!pip install requests scikit-learn beautifulsoup4 nltk google-genai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Categories
posts are sorted into categories and stored in the database - for use by the search interface

In [2]:
import sqlite3
import numpy as np
import re
from collections import defaultdict

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

#colab path
DB_PATH = "/content/drive/MyDrive/globalwarming-arclein.blogspot/Data/insights.db"
#DB_PATH = "../data/insights.db"
# ============================================================
# TOP-LEVEL CATEGORY RULES
# ============================================================

CATEGORY_RULES = {

    "Climate Science": [
        "climate", "warming", "temperature",
        "co2", "carbon", "glacier",
        "ice", "arctic", "atmosphere"
    ],

    "Energy": [
        "energy", "oil", "gas",
        "solar", "wind", "battery",
        "nuclear", "fusion", "reactor"
    ],

    "Agriculture": [
        "agriculture", "crop", "soil",
        "farm", "food", "fertility",
        "irrigation"
    ],

    "Economics": [
        "economy", "economic", "market",
        "finance", "inflation", "debt",
        "trade", "bank"
    ],

    "Geopolitics": [
        "war", "china", "russia",
        "military", "government",
        "conflict", "nato"
    ],

    "Technology": [
        "technology", "ai", "robot",
        "computer", "software",
        "internet", "automation"
    ],

    "Health": [
        "health", "disease", "virus",
        "medical", "vaccine",
        "nutrition"
    ],

    "Environment": [
        "pollution", "forest", "water",
        "ecosystem", "species",
        "biodiversity"
    ],

    "Archaeology": [
        "ancient", "civilization",
        "archaeology", "pyramid",
        "artifact", "historical"
    ],

    "Future Forecasting": [
        "future", "prediction",
        "forecast", "scenario",
        "collapse", "transition"
    ]
}

# ============================================================
# DATABASE HELPERS
# ============================================================

def connect():
    return sqlite3.connect(DB_PATH)


def init_category_tables():
    conn = connect()
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE,
        parent_id INTEGER
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS post_categories (
        post_id INTEGER,
        category_id INTEGER,
        UNIQUE(post_id, category_id)
    )
    """)

    conn.commit()
    conn.close()


# ============================================================
# CATEGORY INSERTION
# ============================================================

def get_or_create_category(name):
    conn = connect()
    cur = conn.cursor()
    cur.execute(
        "SELECT id FROM categories WHERE name=?",
        (name,)
    )
    row = cur.fetchone()
    if row:
        conn.close()
        return row[0]

    cur.execute(
        "INSERT INTO categories (name) VALUES (?)",
        (name,)
    )

    category_id = cur.lastrowid
    conn.commit()
    conn.close()
    return category_id


def assign_post_category(post_id, category_name):
    category_id = get_or_create_category(
        category_name
    )

    conn = connect()
    cur = conn.cursor()

    cur.execute("""
    INSERT OR IGNORE INTO post_categories (
        post_id,
        category_id
    )
    VALUES (?, ?)
    """, (
        post_id,
        category_id
    ))

    conn.commit()
    conn.close()


# ============================================================
# RULE-BASED CLASSIFICATION
# ============================================================

def classify_categories(text):
    text_lower = text.lower()
    matched = []

    for category, keywords in CATEGORY_RULES.items():
        for kw in keywords:
            if kw in text_lower:
                matched.append(category)
                break

    return matched


# ============================================================
# LOAD POSTS + EMBEDDINGS
# ============================================================

def load_posts():
    conn = connect()
    cur = conn.cursor()

    cur.execute("""
    SELECT
        id,
        title,
        content,
        embedding
    FROM posts
    WHERE embedding IS NOT NULL
    """)

    rows = cur.fetchall()
    conn.close()
    posts = []

    for r in rows:
        emb = np.frombuffer(
            r[3],
            dtype=np.float32
        )

        posts.append({
            "id": r[0],
            "title": r[1],
            "content": r[2],
            "embedding": emb
        })

    return posts


# ============================================================
# APPLY RULE-BASED CATEGORIES
# ============================================================

def categorize_posts(posts):

    for p in posts:

        categories = classify_categories(
            p["content"]
        )

        for c in categories:

            assign_post_category(
                p["id"],
                c
            )

    print("Rule-based categories assigned.")


# ============================================================
# MAIN
# ============================================================

def main():
    print("\nINITIALIZING CATEGORY TABLES...\n")
    init_category_tables()
    print("\nLOADING POSTS...\n")
    posts = load_posts()
    print(f"Loaded {len(posts)} posts")
    print("\nASSIGNING RULE-BASED CATEGORIES...\n")
    categorize_posts(posts)

    print("\nDONE.\n")

if __name__ == "__main__":
    main()



INITIALIZING CATEGORY TABLES...


LOADING POSTS...

Loaded 21434 posts

ASSIGNING RULE-BASED CATEGORIES...

Rule-based categories assigned.

DONE.



# Clustering

This script demonstrates how to load posts and perform clustering.

In [3]:
import sqlite3
import numpy as np
from collections import defaultdict
from sklearn.cluster import KMeans

# Define the database path
DB_PATH = "/content/drive/MyDrive/globalwarming-arclein.blogspot/Data/insights.db"

# Database connection helper
def connect():
    return sqlite3.connect(DB_PATH)

# Function to load posts and embeddings
def load_posts():
    conn = connect()
    cur = conn.cursor()

    cur.execute("""
    SELECT
        id,
        title,
        content,
        embedding
    FROM posts
    WHERE embedding IS NOT NULL
    """)

    rows = cur.fetchall()
    conn.close()
    posts = []

    for r in rows:
        emb = np.frombuffer(
            r[3],
            dtype=np.float32
        )

        posts.append({
            "id": r[0],
            "title": r[1],
            "content": r[2],
            "embedding": emb
        })

    return posts

# Function to perform clustering
def cluster_posts(posts, n_clusters=20):
    embeddings = np.array([
        p["embedding"]
        for p in posts
    ])

    print(f"Running KMeans clustering with {n_clusters} clusters...")

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10 # Added to suppress future warning
    )

    cluster_ids = kmeans.fit_predict(
        embeddings
    )

    conn = connect()
    cur = conn.cursor()

    # Clear existing post_clusters to avoid duplicates if run multiple times
    cur.execute("DELETE FROM post_clusters")
    cur.execute("DELETE FROM clusters")
    conn.commit()

    # Insert cluster labels (optional, but good practice)
    for i in range(n_clusters):
        cur.execute("INSERT OR IGNORE INTO clusters (id, label) VALUES (?, ?)", (i, f"Cluster {i}"))
    conn.commit()

    for i, post in enumerate(posts):
        cluster_id = int(cluster_ids[i])
        cur.execute("""
        INSERT INTO post_clusters (
            post_id,
            cluster_id
        )
        VALUES (?, ?)
        """, (
            post["id"],
            cluster_id
        ))

    conn.commit()
    conn.close()

    print("Clusters assigned.")
    return cluster_ids

# Function to inspect clusters
def inspect_clusters(posts, cluster_ids):
    grouped = defaultdict(list)
    for i, cid in enumerate(cluster_ids):
        grouped[cid].append(
            posts[i]["title"]
        )

    print("\n================ CLUSTERS ================\n")

    for cid, titles in sorted(grouped.items()):
        print(f"\nCLUSTER {cid} (contains {len(titles)} posts)")
        for t in titles[:5]: # Display top 5 titles for brevity
            print("  -", t[:120])


def run_clustering_script():
    print("\nLOADING POSTS...\n")
    posts = load_posts()
    print(f"Loaded {len(posts)} posts")

    # Adjust n_clusters as needed
    cluster_ids = cluster_posts(posts, n_clusters=20)

    inspect_clusters(posts, cluster_ids)
    print("\nClustering script finished.\n")

    ###########
    #find_related_posts(

if __name__ == "__main__":
    run_clustering_script()


LOADING POSTS...

Loaded 21434 posts
Running KMeans clustering with 20 clusters...
Clusters assigned.

================ CLUSTERS ================


CLUSTER 0 (contains 923 posts)
  - efficient incineration
  - Pushing the envelope on incineration
  - Transportation Energy.
  - 100 miles per gallon
  - Celluose conversion

CLUSTER 1 (contains 1405 posts)
  - Jerry Pournelle on Wealth Allocation
  - Chad Charcoal Emergency
  - Global Warming for Dummies
  - Global Warming Business Lobby
  - Capitalism and Crisis

CLUSTER 2 (contains 1239 posts)
  - Pleistocene Nonconformity - 9 - Velikovsky brain candy
  - Lukewarm Fusion
  - Sunspot Genesis
  - Earth's Magnetic Field
  - UFO Enigma

CLUSTER 3 (contains 1002 posts)
  - Golden Mean and Cognition
  - Midday Nap Markedly Boosts Cognitive Ability
  - Autism's Extraordinary Perception
  - Cooking Shaped Human Development
  - Heaven and Science

CLUSTER 4 (contains 1400 posts)
  - EPA Demonstrates Common Sense
  - Bill Quigley's Post on Katri

# Finding Related Posts

This script focuses on loading posts from the database and using the `find_related_posts` function to identify similar content.

In [ ]:
# This script load all posts and their embeddings directly from the database and then calculate related posts.

import sqlite3
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Define the database path
DB_PATH = "/content/drive/MyDrive/globalwarming-arclein2/clustering/insights.db"

# Database connection helper
def connect():
    return sqlite3.connect(DB_PATH)

# Function to initialize all necessary tables (copied for self-contained script)
def init_db_tables():
    conn = connect()
    cur = conn.cursor()

    # Create posts table (required for loading data)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS posts (
        id INTEGER PRIMARY KEY,
        title TEXT,
        content TEXT,
        embedding BLOB
    )
    """)

    # Create categories table (needed if using other functions, but good to have complete init)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE,
        parent_id INTEGER
    )
    """)

    # Create post_categories table
    cur.execute("""
    CREATE TABLE IF NOT EXISTS post_categories (
        post_id INTEGER,
        category_id INTEGER,
        UNIQUE(post_id, category_id)
    )
    """)

    conn.commit()
    conn.close()
    print("Database tables initialized (or already exist).")

# Function to load posts and embeddings (copied for self-contained script)
def load_posts():
    conn = connect()
    cur = conn.cursor()

    cur.execute("""
    SELECT
        id,
        title,
        content,
        embedding
    FROM posts
    WHERE embedding IS NOT NULL
    """)

    rows = cur.fetchall()
    conn.close()
    posts = []

    for r in rows:
        emb = np.frombuffer(
            r[3],
            dtype=np.float32
        )

        posts.append({
            "id": r[0],
            "title": r[1],
            "content": r[2],
            "embedding": emb
        })

    return posts

# Function to find related posts (copied from previous cell)
def find_related_posts(posts, top_k=5):
    if not posts:
        return {}

    embeddings = np.array([
        p["embedding"]
        for p in posts
    ])

    sim = cosine_similarity(
        embeddings
    )

    related = {}

    for i, p in enumerate(posts):

        scores = list(
            enumerate(sim[i])
        )

        # Sort by similarity score in descending order
        scores = sorted(
            scores,
            key=lambda x: x[1],
            reverse=True
        )

        related_posts = []

        # Skip the first element as it's the post itself (score 1.0)
        for idx, score in scores[1:top_k+1]:

            related_posts.append({
                "post_id": posts[idx]["id"],
                "title": posts[idx]["title"],
                "score": float(score)
            })

        related[p["id"]] = related_posts

    return related

def run_related_posts_script():
    print("\nINITIALIZING DATABASE TABLES...\n")
    #init_db_tables() # Ensure tables exist

    print("\nLOADING POSTS...\n")
    posts = load_posts()
    print(f"Loaded {len(posts)} posts with embeddings.")

    if posts:
        print("\nGENERATING RELATED POSTS...\n")
        related = find_related_posts(
            posts,
            top_k=5
        )

        if related:
            print("\nEXAMPLE RELATED POSTS FOR A RANDOM POST:\n")
            # Get a random post ID to show related posts
            first_post_id = list(related.keys())[0]
            original_post_title = next(p['title'] for p in posts if p['id'] == first_post_id)
            print(f"Original Post: {original_post_title}\n")

            for r in related[first_post_id]:
                print(
                    f"  - Related: {r['title']}\n    SCORE: {round(r['score'], 3)}"
                )
        else:
            print("No related posts found (possibly due to lack of posts or embeddings).")
    else:
        print("No posts loaded from the database to find related items.")

    print("\nRelated posts script finished.\n")

if __name__ == "__main__":
    run_related_posts_script()

#Export updated posts, categories, clusters, etc to posts.json

In [4]:
#find_related_posts
# export database to posts.json
# this script exports categories and clusters

import sqlite3
import json
import numpy as np
from collections import defaultdict

#DB_PATH = "../data/insights.db"
DB_PATH = "/content/drive/MyDrive/globalwarming-arclein.blogspot/Data/insights.db"

# ============================================================
# DATABASE
# ============================================================

def connect():
    return sqlite3.connect(DB_PATH)


# ============================================================
# LOAD CATEGORY MAP
# ============================================================

def load_categories():

    conn = connect()

    cur = conn.cursor()

    cur.execute("""

    SELECT
        pc.post_id,
        c.name

    FROM post_categories pc

    JOIN categories c
        ON pc.category_id = c.id

    """)

    rows = cur.fetchall()

    conn.close()

    category_map = defaultdict(list)

    for post_id, category in rows:

        category_map[post_id].append(category)

    return category_map


# ============================================================
# LOAD CLUSTERS
# ============================================================

def load_clusters():

    conn = connect()

    cur = conn.cursor()

    cur.execute("""

    SELECT
        pc.post_id,
        pc.cluster_id,
        c.label

    FROM post_clusters pc

    LEFT JOIN clusters c
        ON pc.cluster_id = c.id

    """)

    rows = cur.fetchall()

    conn.close()

    cluster_map = {}

    for post_id, cluster_id, label in rows:

        cluster_map[post_id] = {
            "id": cluster_id,
            "label": label
        }

    return cluster_map


# ============================================================
# LOAD RELATED POSTS
# ============================================================

def load_related_posts():

    conn = connect()

    cur = conn.cursor()

    # OPTIONAL TABLE:
    # related_posts(
    #   post_id,
    #   related_post_id,
    #   similarity
    # )

    try:

        cur.execute("""

        SELECT
            rp.post_id,
            rp.related_post_id,
            rp.similarity,
            p.title

        FROM related_posts rp

        JOIN posts p
            ON rp.related_post_id = p.id

        """)

        rows = cur.fetchall()

    except:

        rows = []

    conn.close()

    related_map = defaultdict(list)

    for post_id, related_id, sim, title in rows:

        related_map[post_id].append({

            "id": related_id,

            "title": title,

            "similarity": round(sim, 3)
        })

    return related_map


# ============================================================
# SIMPLE SUBTOPIC EXTRACTION
# ============================================================

def generate_subtopics(tags):

    # For now:
    # reuse top tags as subtopics

    return tags[:3]


# ============================================================
# WORD COUNT
# ============================================================

def word_count(text):

    return len(text.split())


# ============================================================
# READING TIME
# ============================================================

def reading_time_minutes(text):

    wc = word_count(text)

    return max(1, round(wc / 250))


# ============================================================
# EXPORT JSON
# ============================================================

def export_json():

    print("\nLOADING SUPPORTING DATA...\n")

    category_map = load_categories()

    cluster_map = load_clusters()

    related_map = load_related_posts()

    print("\nLOADING POSTS...\n")

    conn = connect()

    cur = conn.cursor()

    cur.execute("""

    SELECT
        id,
        url,
        title,
        date,
        content,
        summary,
        insight,
        tags,
        embedding

    FROM posts

    """)

    rows = cur.fetchall()

    conn.close()

    data = []

    total = len(rows)

    print(f"PROCESSING {total} POSTS...\n")

    for i, r in enumerate(rows):
        post_id = r[0]
        url = r[1]
        title = r[2]
        date = r[3]
        content = r[4] or ""
        summary = r[5] or ""
        insight = r[6] or ""
        tags_str = r[7] or ""
        embedding_blob = r[8]

        tags = [
            t.strip()
            for t in tags_str.split(",")
            if t.strip()
        ]

        categories = category_map.get(
            post_id,
            []
        )

        cluster = cluster_map.get(
            post_id,
            {
                "id": None,
                "label": None
            }
        )

        related_posts = related_map.get(
            post_id,
            []
        )

        subtopics = generate_subtopics(
            tags
        )

        # embedding metadata only
        embedding_dimension = None

        if embedding_blob:

            emb = np.frombuffer(
                embedding_blob,
                dtype=np.float32
            )

            embedding_dimension = len(emb)

        post_json = {
            "id": post_id,
            "url": url,
            "title": title,
            "date": date,
            "summary": summary,
            "insight": insight,
            "categories": categories,
            "subtopics": subtopics,
            "tags": tags,
            "cluster": {
                "id": cluster["id"],
                "label": cluster["label"]
            },
            "related_posts": related_posts,
            "entities": {
                "locations": [],
                "organizations": [],
                "people": [],
                "technologies": []
            },
            "metrics": {
                "word_count": word_count(
                    content
                ),
                "reading_time_minutes":
                    reading_time_minutes(
                        content
                    )
            },
            "search": {
                "embedding_model":
                    "all-MiniLM-L6-v2",
                "embedding_dimension":
                    embedding_dimension
            },
            "pipeline": {
                "version": "2.0"
            }
        }
        data.append(post_json)

        if i % 1000 == 0 and i > 0:
            print(
                f"Processed {i}/{total}"
            )

    print("\nWRITING JSON FILE...\n")

    #"../exports/posts.json",
    with open(
        "posts.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"\nEXPORTED {len(data)} POSTS"
    )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    export_json()


LOADING SUPPORTING DATA...


LOADING POSTS...

PROCESSING 21434 POSTS...

Processed 1000/21434
Processed 2000/21434
Processed 3000/21434
Processed 4000/21434
Processed 5000/21434
Processed 6000/21434
Processed 7000/21434
Processed 8000/21434
Processed 9000/21434
Processed 10000/21434
Processed 11000/21434
Processed 12000/21434
Processed 13000/21434
Processed 14000/21434
Processed 15000/21434
Processed 16000/21434
Processed 17000/21434
Processed 18000/21434
Processed 19000/21434
Processed 20000/21434
Processed 21000/21434

WRITING JSON FILE...


EXPORTED 21434 POSTS


# Map categories
map categories, etc, from updated posts.json to clean_labels.json (this file is used for the search interface)

In [5]:
#map_categories
import collections
import json
import os

input_filename = "/content/drive/MyDrive/globalwarming-arclein.blogspot/Exports/posts.json"
output_filename = "/content/drive/MyDrive/globalwarming-arclein.blogspot/Exports/clean_labels.json"

# Verification step
if not os.path.exists(input_filename):
    print(f"Error: Could not find '{input_filename}' in this folder.")
    print("Please make sure the script is running in the same directory as your JSON file.")
    exit()

print(f"Success! Found '{input_filename}'. Mapping posts by their actual categories array...")

try:
    with open(input_filename, "r", encoding="utf-8") as f:
        posts = json.load(f)

    # Dictionary to group articles by their clean categories
    categories_map = collections.defaultdict(list)
    processed_count = 0

    for post in posts:
        title = post.get("title", "Untitled").strip()

        # Note: Your JSON uses a backend URL pattern, if it's missing we default to '#'
        url = post.get("url", "#").strip()

        # TARGET THE EXACT MATCH: The "categories" list array from your file
        post_categories = post.get("categories", [])

        if post_categories and isinstance(post_categories, list):
            processed_count += 1
            for category in post_categories:
                category_clean = category.strip()
                if category_clean:
                    # Append the lightweight search data needed for your menu layout
                    categories_map[category_clean].append({
                        "title": title,
                        "url": url
                    })

    # Output the optimized reverse lookup dictionary map
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(categories_map, f, indent=2, ensure_ascii=False)

    print(f"\nSuccess! Successfully processed {len(posts)} items.")
    print(f"Extracted {len(categories_map)} true structural categories.")
    print(f"Optimized category navigation written to: '{output_filename}'")

    # Print your final clean structural list
    print("\nYour True Blog Categories Found:")
    for cat in sorted(categories_map.keys()):
        print(f" - {cat} ({len(categories_map[cat])} articles)")

except json.JSONDecodeError:
    print(f"Error: '{input_filename}' has formatting issues. It is not valid JSON.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Success! Found '/content/drive/MyDrive/globalwarming-arclein.blogspot/Exports/posts.json'. Mapping posts by their actual categories array...

Success! Successfully processed 21448 items.
Extracted 0 true structural categories.
Optimized category navigation written to: '/content/drive/MyDrive/globalwarming-arclein.blogspot/Exports/clean_labels.json'

Your True Blog Categories Found:
